# Week 4 Lab 3 — DeepONet-Style Cavity Field Surrogate

**Runtime:** Colab T4 GPU recommended.  
The branch receives Reynolds number and the trunk receives spatial coordinates. This teaches the branch--trunk architecture while preserving an honest scientific claim: the current dataset supports parameter-to-field learning, not arbitrary function-to-function learning.


In [ ]:
from pathlib import Path
import time, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from cavity_project_utils import load_dataset,field_errors,save_predictions

SEED=690; np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
data=load_dataset("combined_cavity_dataset.npz")
train=data["split"]=="train"; test=data["split"]=="test"
rmin,rmax=data["Re"][train].min(),data["Re"][train].max()
scale_r=lambda r: 2*(r-rmin)/(rmax-rmin)-1
Xg,Yg=np.meshgrid(data["x"],data["y"])
print("train",data["Re"][train],"test",data["Re"][test])


## 1. Architecture

For latent width (p), the branch returns two sets of coefficients, one for (u) and one for (v). The trunk returns spatial basis features. Inner products produce the two velocity components.


In [ ]:
LATENT=48

branch_in=tf.keras.Input((1,),name="branch_Re")
b=tf.keras.layers.Dense(64,activation="tanh")(branch_in)
b=tf.keras.layers.Dense(64,activation="tanh")(b)
b=tf.keras.layers.Dense(2*LATENT)(b)
b=tf.keras.layers.Reshape((2,LATENT))(b)

trunk_in=tf.keras.Input((2,),name="trunk_xy")
t=tf.keras.layers.Dense(64,activation="tanh")(trunk_in)
t=tf.keras.layers.Dense(64,activation="tanh")(t)
t=tf.keras.layers.Dense(LATENT)(t)

out=tf.keras.layers.Lambda(lambda z: tf.einsum("bcp,bp->bc",z[0],z[1]),name="branch_trunk_dot")([b,t])
deeponet=tf.keras.Model([branch_in,trunk_in],out,name="Parameterized_DeepONet_Cavity")
deeponet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="mse")
deeponet.summary()


## 2. Build leakage-free training rows


In [ ]:
def samples(mask,stride=2):
    xx=Xg[::stride,::stride].ravel(); yy=Yg[::stride,::stride].ravel()
    B=[];T=[];Q=[]
    for k in np.where(mask)[0]:
        B.append(np.full((len(xx),1),scale_r(data["Re"][k])))
        T.append(np.c_[2*xx-1,2*yy-1])
        Q.append(np.c_[data["u"][k,::stride,::stride].ravel(),data["v"][k,::stride,::stride].ravel()])
    return np.vstack(B).astype("float32"),np.vstack(T).astype("float32"),np.vstack(Q).astype("float32")

Btr,Ttr,Qtr=samples(train,stride=2)
print(Btr.shape,Ttr.shape,Qtr.shape)


## 3. Train


In [ ]:
t0=time.time()
hist=deeponet.fit([Btr,Ttr],Qtr,epochs=700,batch_size=1024,verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="loss",patience=60,restore_best_weights=True)])
train_seconds=time.time()-t0
print("epochs",len(hist.history["loss"]),"seconds",train_seconds,"parameters",deeponet.count_params())
plt.semilogy(hist.history["loss"]);plt.xlabel("epoch");plt.ylabel("training MSE");plt.grid(alpha=.3);plt.show()


## 4. Frozen blind evaluation


In [ ]:
def predict_deeponet(Re):
    n=Xg.size
    B=np.full((n,1),scale_r(Re),dtype="float32")
    T=np.c_[2*Xg.ravel()-1,2*Yg.ravel()-1].astype("float32")
    q=deeponet.predict([B,T],verbose=0)
    N=len(data["x"]); return q[:,0].reshape(N,N),q[:,1].reshape(N,N)

results=[]
for idx in np.where(test)[0]:
    up,vp=predict_deeponet(data["Re"][idx])
    results.append((float(data["Re"][idx]),field_errors(data["u"][idx],data["v"][idx],up,vp)))
results


In [ ]:
idx=np.where(test)[0][0]; Re_star=data["Re"][idx]
up,vp=predict_deeponet(Re_star)
dx=data["x"][1]-data["x"][0];dy=data["y"][1]-data["y"][0]
div=np.gradient(up,dx,axis=1)+np.gradient(vp,dy,axis=0)
physics={"div_l2":float(np.sqrt(np.mean(div[1:-1,1:-1]**2))),
         "lid_u_MAE":float(np.mean(np.abs(up[-1]-1))),
         "stationary_wall_max":float(max(np.hypot(up[0],vp[0]).max(),np.hypot(up[:,0],vp[:,0]).max(),np.hypot(up[:,-1],vp[:,-1]).max()))}
print(json.dumps(physics,indent=2))
save_predictions("predictions_deeponet.npz",Re_star,data["x"],data["y"],data["u"][idx],data["v"][idx],up,vp,"parameterized_deeponet")


In [ ]:
err=np.hypot(up-data["u"][idx],vp-data["v"][idx])
mid=len(data["x"])//2
fig,ax=plt.subplots(2,2,figsize=(10,8))
ax[0,0].streamplot(Xg,Yg,up,vp,density=1.1);ax[0,0].set_title(f"DeepONet Re={Re_star:g}")
im=ax[0,1].contourf(Xg,Yg,err,30);fig.colorbar(im,ax=ax[0,1]);ax[0,1].set_title("vector error")
ax[1,0].plot(data["u"][idx,:,mid],data["y"],label="CFD");ax[1,0].plot(up[:,mid],data["y"],"--",label="DeepONet");ax[1,0].legend();ax[1,0].set(xlabel="u",ylabel="y")
ax[1,1].plot(data["x"],data["v"][idx,mid,:],label="CFD");ax[1,1].plot(data["x"],vp[mid,:],"--",label="DeepONet");ax[1,1].legend();ax[1,1].set(xlabel="x",ylabel="v")
plt.tight_layout();plt.show()


## 5. Required comparison and interpretation

Create one table comparing field interpolation, coordinate DNN, and DeepONet on every blind Reynolds number. Use identical metrics.

Answer:

1. What does the branch encode? What does the trunk encode?  
2. Why is a scalar-Re branch not full function-to-function operator learning?  
3. Propose a true operator dataset with branch vector ([U_{lid}(\xi_1),\ldots,U_{lid}(\xi_m)]).  
4. Does DeepONet improve blind field error or physical metrics?  
5. Which spatial zone controls the largest error?  
6. What physics loss would you add in Week 5, and what failure could it introduce if weighted poorly?
